# CMI and Markov length for a ring of toric plaquettes — optimized version

This notebook is a memory-safe rewrite of `toric_code_alpha.ipynb`.

The original notebook crashed because it explicitly built dense objects of size
\(2^N\times 2^N\), for example `rho_0`, plaquette operators, Pauli-Z operators, and dense partial traces.

This version **never constructs the full density matrix**.  Instead it uses the fact that the state

\[
|\psi_0\rangle = \prod_p \frac{I+\alpha A_p}{\sqrt{1+\alpha^2}} |0\rangle^{\otimes N}
\]

has support only on the bitstrings generated by products of plaquette flips.  For `m` plaquettes this is at most \(2^m\) states, rather than \(2^N\) basis states.

The phase-flip channel is also applied analytically: for a density matrix element
\(|s\rangle\langle t|\), dephasing on one qubit multiplies it by `1` if the two bitstrings agree on that qubit and by `1 - 2p` if they differ.  This avoids dense `Z_i @ rho @ Z_i` operations.


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from functools import lru_cache
from pathlib import Path
from concurrent.futures import ProcessPoolExecutor, as_completed

# Optional style; the notebook works even if scienceplots is not installed.
try:
    import scienceplots
    plt.style.use('science')
except Exception:
    pass

Path('New_plots').mkdir(exist_ok=True)


## Parameters

In [ ]:
# Weight parameter in (I + alpha A_p) / sqrt(1 + alpha^2)
alpha = 0.68


## Sparse bitstring representation

A computational basis state is stored as a Python integer.  Qubit `q` corresponds to bit `q`.  A plaquette operator `A_p = XXXX` acts by flipping the four bits in its plaquette mask.


In [ ]:
def mask_from_qubits(qubits):
    """Integer bit mask with ones on the listed qubits."""
    mask = 0
    for q in qubits:
        mask |= (1 << int(q))
    return mask


def get_plaquettes(number_of_plaquettes):
    """Plaquette qubit indices for the ring geometry used in the original notebook."""
    N = 3 * int(number_of_plaquettes)
    return [
        ((0 + 3*n) % N, (1 + 3*n) % N, (2 + 3*n) % N, (3 + 3*n) % N)
        for n in range(int(number_of_plaquettes))
    ]


def get_error_index(total_system_size):
    """Qubits on which the phase-flip channel is applied."""
    return [i for i in range(int(total_system_size)) if i % 3 == 0]


print('4-plaquette ring:', get_plaquettes(4))
print('Error indices for N=12:', get_error_index(12))


In [ ]:
@lru_cache(maxsize=None)
def sparse_ground_state_cached(N, plaquettes_tuple, alpha_value):
    """
    Return the ground-state support and amplitudes without constructing operators.

    Output:
        states: integer bitstrings with nonzero amplitude
        amps:   corresponding real amplitudes
    """
    plaquettes = [tuple(p) for p in plaquettes_tuple]
    norm = np.sqrt(1.0 + alpha_value**2)

    # dictionary: bitstring -> amplitude
    state = {0: 1.0}

    for p in plaquettes:
        flip = mask_from_qubits(p)
        new_state = {}
        for s, a in state.items():
            # Identity branch
            new_state[s] = new_state.get(s, 0.0) + a / norm
            # Plaquette-flipped branch
            sf = s ^ flip
            new_state[sf] = new_state.get(sf, 0.0) + alpha_value * a / norm
        state = new_state

    states = np.array(list(state.keys()), dtype=object)  # object keeps arbitrary-size Python ints safe
    amps = np.array(list(state.values()), dtype=np.float64)

    # Remove tiny roundoff entries and normalize defensively.
    keep = np.abs(amps) > 1e-15
    states = states[keep]
    amps = amps[keep]
    amps = amps / np.linalg.norm(amps)
    return states, amps


def sparse_ground_state(N, plaquettes, alpha_value=alpha):
    plaquettes_tuple = tuple(tuple(map(int, p)) for p in plaquettes)
    return sparse_ground_state_cached(int(N), plaquettes_tuple, float(alpha_value))


states_12, amps_12 = sparse_ground_state(12, get_plaquettes(4), alpha)
print('Number of full Hilbert-space basis states for N=12:', 2**12)
print('Number of actually occupied states:', len(states_12))
print('Norm:', np.sum(np.abs(amps_12)**2))


## Reduced density matrices without full density matrices

For a subsystem `keep`, the reduced matrix element is assembled from pairs of sparse-support states that agree on the traced-out environment.  The returned matrix is compact: its basis contains only subsystem bitstrings that actually occur in the sparse support.  This is important when `keep` is large, for example when `keep = ABC` is the whole system.


In [ ]:
def compress_bits(x, qubits):
    """Extract selected qubits from integer x and pack them into consecutive bits."""
    y = 0
    for out_pos, q in enumerate(qubits):
        if (int(x) >> int(q)) & 1:
            y |= (1 << out_pos)
    return y


def reduced_density_matrix_sparse(states, amps, keep, N, noise_indices=None, p=0.0):
    """
    Reduced density matrix after independent phase-flip noise, computed only on sparse support.

    Args:
        states, amps: sparse pure-state representation before noise.
        keep: subsystem qubits to keep.
        N: total number of qubits.
        noise_indices: qubits affected by phase-flip channel.
        p: phase-flip probability.

    Returns:
        rho_red: compact reduced density matrix.  Its dimension is the number of
                 distinct kept-subsystem bitstrings appearing in the sparse support,
                 not necessarily 2**len(keep).
    """
    keep = sorted(set(map(int, keep)))
    env = [q for q in range(int(N)) if q not in keep]
    noise_mask = mask_from_qubits([] if noise_indices is None else noise_indices)
    gamma = 1.0 - 2.0 * float(p)

    # Precompute compressed subsystem/environment labels.
    klabels_raw = [compress_bits(s, keep) for s in states]
    elabels = [compress_bits(s, env) for s in states]

    # Compact subsystem basis: raw kept bitstring -> dense matrix index.
    unique_k = sorted(set(klabels_raw))
    k_to_compact = {k: i for i, k in enumerate(unique_k)}
    klabels = np.array([k_to_compact[k] for k in klabels_raw], dtype=np.int64)

    # Group support states by identical environment label; only those contribute after tracing env out.
    groups = {}
    for idx, e in enumerate(elabels):
        groups.setdefault(e, []).append(idx)

    dim = len(unique_k)
    rho = np.zeros((dim, dim), dtype=np.complex128)

    for idxs in groups.values():
        # The group sizes are typically small for local subsystems, but this also works for keep=all.
        for ia in idxs:
            s = int(states[ia])
            a = amps[ia]
            ka = klabels[ia]
            for ib in idxs:
                t = int(states[ib])
                b = amps[ib]
                kb = klabels[ib]

                # Phase-flip channel factor: product over noisy qubits.
                ndiff = ((s ^ t) & noise_mask).bit_count()
                factor = gamma ** ndiff
                rho[ka, kb] += a * np.conjugate(b) * factor

    # Numerical cleanup.
    rho = 0.5 * (rho + rho.conjugate().T)
    tr = np.trace(rho).real
    if tr > 0:
        rho /= tr
    return rho


def von_neumann_entropy_from_rho(rho, cutoff=1e-12):
    eigvals = np.linalg.eigvalsh(rho)
    eigvals = eigvals[eigvals > cutoff]
    return float(-np.sum(eigvals * np.log2(eigvals)))


def entropy_sparse(states, amps, keep, N, noise_indices, p):
    rho_red = reduced_density_matrix_sparse(states, amps, keep, N, noise_indices, p)
    return von_neumann_entropy_from_rho(rho_red)


def CMI_sparse(p, noise_indices, states, amps, A, B, C, N):
    """Compute I(A:C|B) = S(AB) + S(BC) - S(B) - S(ABC)."""
    A = list(map(int, A)); B = list(map(int, B)); C = list(map(int, C))
    AB = sorted(set(A + B))
    BC = sorted(set(B + C))
    ABC = sorted(set(A + B + C))
    B = sorted(set(B))

    S_AB = entropy_sparse(states, amps, AB, N, noise_indices, p)
    S_BC = entropy_sparse(states, amps, BC, N, noise_indices, p)
    S_B = entropy_sparse(states, amps, B, N, noise_indices, p)
    S_ABC = entropy_sparse(states, amps, ABC, N, noise_indices, p)
    return S_AB + S_BC - S_B - S_ABC


## CMI for the 12-qubit / 4-plaquette example

In [ ]:
# Regions from the original notebook
A = [1, 2]
B = [3, 4, 5, 6, 9, 10, 11, 0]
C = [7, 8]

N = 12
plaquettes_4 = get_plaquettes(4)
noise_indices_4 = get_error_index(N)
states_4, amps_4 = sparse_ground_state(N, plaquettes_4, alpha)

p_array = np.linspace(0.001, 0.999, 40)
cmi_array = np.array([
    CMI_sparse(p, noise_indices_4, states_4, amps_4, A, B, C, N)
    for p in p_array
])

print(cmi_array)


In [ ]:
fig, ax = plt.subplots(figsize=(6, 3))
ax.plot(p_array, cmi_array, lw=1)
ax.scatter(p_array, cmi_array, color='red', label=r'$I(A{:}C|B)$')
ax.axvline(0.5, color='darkblue', ls=':', alpha=0.4)
ax.set_xlabel(r'Noise rate $p$', fontsize=12)
ax.set_ylabel(r'$I(A{:}C \mid B)$', fontsize=12)
ax.set_title(r'$I(A{:}C|B)$ vs $p$', fontsize=12)
ax.legend(fontsize=10)
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig('New_plots/CMI_vs_p_plaquette_ring_optimized.png', dpi=300)
plt.show()


## Markov length computation

The function below preserves the partitioning logic from the original notebook but replaces the dense density-matrix calculation with `CMI_sparse`.

Increase `R_max` gradually.  The memory bottleneck is now the number of plaquette-generated support states and the compact reduced-density dimensions, not the full \(2^N\times 2^N\) density matrix.


In [ ]:
def partition_for_distance(d):
    """Original distance-dependent A, B, C construction."""
    d = int(d)
    A_d = list(range(1, 3))
    B_d = list(range(0, 1)) + list(range(3, 3 + d*3 + 1)) + list(range(3 + d*3 + 3, 3 + d*3 + 3 + d*3))
    C_d = list(range(3 + d*3 + 1, 3 + d*3 + 3))
    N_d = max(A_d + B_d + C_d) + 1
    return A_d, B_d, C_d, N_d


def cmi_for_distance(p, d, alpha_value=alpha):
    A_d, B_d, C_d, N_d = partition_for_distance(d)
    nb_plaquettes = N_d // 3
    states, amps = sparse_ground_state(N_d, get_plaquettes(nb_plaquettes), alpha_value)
    noise_indices = get_error_index(N_d)
    return CMI_sparse(p, noise_indices, states, amps, A_d, B_d, C_d, N_d)


def fit_markov_length_offset(distances, cmi_values, offset=0.0, threshold=1e-12):
    distances = np.asarray(distances, dtype=float)
    cmi_values = np.asarray(cmi_values, dtype=float)
    shifted = cmi_values - offset
    mask = shifted > threshold
    if np.sum(mask) < 2:
        return np.nan
    slope, intercept = np.polyfit(distances[mask], np.log(shifted[mask]), 1)
    if slope >= 0:
        return np.nan
    return -1.0 / slope


In [ ]:
# Noise rates used to evaluate Markov length
p_array2 = np.linspace(0.001, 0.999, 10)

# Increase this gradually.  The optimized code can handle larger values than the dense version,
# but the problem is still exponential in the number of plaquette-generated configurations.
R_max = 2
distances = np.arange(1, R_max + 1)

cmi_matrix = np.zeros((len(p_array2), len(distances)))
for ip, p in enumerate(p_array2):
    for idd, d in enumerate(distances):
        cmi_matrix[ip, idd] = cmi_for_distance(p, d, alpha)

xi_array = np.array([
    fit_markov_length_offset(distances, cmi_matrix[ip], offset=0.0)
    for ip in range(len(p_array2))
])

print('cmi_matrix:')
print(cmi_matrix)
print('xi_array:')
print(xi_array)
print('finite xi values:', np.sum(np.isfinite(xi_array)))


## Optional parallel version for larger parameter grids

Use this cell instead of the serial Markov-length cell above when the grid of `p` and `r` values becomes large.  On Windows/macOS notebooks, multiprocessing can sometimes be less stable than serial execution; if it causes issues, keep `USE_PARALLEL = False`.

In [ ]:
USE_PARALLEL = False
MAX_WORKERS = None  # None means Python chooses a sensible default.

if USE_PARALLEL:
    jobs = [(ip, idd, float(p), int(d)) for ip, p in enumerate(p_array2) for idd, d in enumerate(distances)]
    cmi_matrix_parallel = np.zeros((len(p_array2), len(distances)))

    with ProcessPoolExecutor(max_workers=MAX_WORKERS) as ex:
        future_to_job = {
            ex.submit(cmi_for_distance, p, d, alpha): (ip, idd, p, d)
            for ip, idd, p, d in jobs
        }
        for fut in as_completed(future_to_job):
            ip, idd, p, d = future_to_job[fut]
            cmi_matrix_parallel[ip, idd] = fut.result()

    cmi_matrix = cmi_matrix_parallel
    xi_array = np.array([
        fit_markov_length_offset(distances, cmi_matrix[ip], offset=0.0)
        for ip in range(len(p_array2))
    ])

    print('Parallel computation complete.')
    print(cmi_matrix)
    print(xi_array)


## Plots

In [ ]:
valid = np.isfinite(xi_array)

plt.figure(figsize=(8, 5))
plt.plot(p_array2, xi_array, marker='o', lw=2)
plt.axvline(0.5, color='darkblue', ls='-', alpha=0.4)
plt.xlabel(r'Noise rate $p$')
plt.ylabel(r'Markov length $\xi(p)$')
plt.title(r'Markov length from decay of $I(A:C|B)$')
plt.tight_layout()
plt.grid(True)
plt.savefig('New_plots/Xi_vs_p_plaquette_ring_optimized.png', dpi=300)
plt.show()


In [ ]:
plt.figure(figsize=(8, 5))
for idd, r in enumerate(distances):
    plt.plot(p_array2, cmi_matrix[:, idd], marker='o', label=fr'$r={r}$')

plt.xlabel('Noise rate $p$')
plt.ylabel(r'$I(A:C|B)$')
plt.title('CMI vs noise rate for different distances')
plt.legend()
plt.grid(True)
plt.tight_layout()
plt.savefig('New_plots/CMI_vs_p_different_r_plaquette_ring_optimized.png', dpi=300)
plt.show()


In [ ]:
plt.figure(figsize=(8, 6))
for idx in range(1, len(p_array2), 2):
    y = cmi_matrix[idx]
    mask = y > 1e-12
    if np.any(mask):
        plt.plot(distances[mask], y[mask], marker='o', label=fr'$p={p_array2[idx]:.3f}$')

plt.yscale('log')
plt.xlabel('Distance $r$')
plt.ylabel(r'$I(A:C|B)$')
plt.title('Exponential decay of CMI with distance')
plt.legend(loc='upper right')
plt.tight_layout()
plt.savefig('New_plots/CMI_decay_with_r_plaquette_ring_optimized.png', dpi=300)
plt.show()


## Scaling notes

This version should not crash from trying to allocate dense full-system matrices.  However, the computation is still exponential in the number of independent plaquettes because the exact ground state has up to \(2^{N/3}\) support states.

Practical advice:

- Increase `R_max` gradually.
- Keep regions `A`, `B`, and `C` as small as physically reasonable.
- Use the parallel cell for large parameter sweeps, not for a single huge matrix.
- If this still becomes slow for very large `N`, the next step is a tensor-network contraction or a batched PyTorch/JAX implementation for the compact reduced matrices.
